# Лабораторная работа: Создание диалога агентов на основе LLM

## Цель работы
Познакомиться с базовым использованием языковых моделей (LLM) через библиотеку `transformers` и реализовать простого агента, который может вести диалог.

## Что нужно сделать
В коде ниже некоторые строки **удалены** (заменены на комментарии `# ЗАДАНИЕ №N: ...`).  
Ваша задача — восстановить недостающий код.  
Логика программы не меняется — нужно только правильно заполнить пропуски.

## Рекомендации
- Читайте комментарии внимательно — в них есть подсказки.
- Запускайте ячейки **по порядку**.
- Если появляется ошибка — проверьте, правильно ли вы заполнили предыдущие задания.
- Путь к модели оставьте как есть (или укажите свой, если модель лежит в другом месте).

---


In [ ]:
# Установка необходимых библиотек (запустите один раз)
pip install torch transformers

## Часть 1. Простая генерация ответа моделью

Сначала научимся загружать модель и получать от неё простой ответ.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# ВАЖНО: СКАЧАЙТЕ МОДЕЛЬ
#
# 1. Выберите небольшую модель (рекомендуется для слабых ПК):
#    - Qwen/Qwen2.5-0.5B-Instruct
#    - Qwen/Qwen2.5-1.5B-Instruct
#    ...
#
# 2. Скачайте её ...
# 3. Укажите ПУТЬ к папке с моделью ниже (или имя модели с Hugging Face).
model_path = r"ПУТЬ_К_ВАШЕЙ_МОДЕЛИ"   # например: r"C:\Users\Student\Downloads\Qwen2.5-0.5B"
                                     # или просто: "Qwen/Qwen2.5-0.5B-Instruct"

# ============================================================
# ЗАДАНИЕ №1
# Загрузите токенизатор и модель.
# Подсказка:
#   tokenizer = AutoTokenizer.from_pretrained(...)
#   model = AutoModelForCausalLM.from_pretrained(..., torch_dtype="auto", device_map="auto")
# ============================================================

# >>> ВАШ КОД ЗДЕСЬ (2 строки) <<<



# Функция для генерации ответа
def generate_response(prompt, max_length=512):
    # ============================================================
    # ЗАДАНИЕ №2
    # Преобразуйте текст prompt в тензоры (токены) и отправьте их на устройство модели.
    # Подсказка: inputs = tokenizer(..., return_tensors="pt").to(model.device)
    # ============================================================
    
    # >>> ВАШ КОД ЗДЕСЬ (1 строка) <<<
    
    
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # ============================================================
    # ЗАДАНИЕ №3
    # Декодируйте сгенерированные токены обратно в текст.
    # Подсказка: response = tokenizer.decode(..., skip_special_tokens=True)
    # ============================================================
    
    # >>> ВАШ КОД ЗДЕСЬ (1 строка) <<<
    
    
    return response

# Проверка: пример использования
prompt = "Объясни, что такое искусственный интеллект простыми словами."
response = generate_response(prompt)
print("Ответ модели:", response)

## Часть 2. Класс Agent — агент с памятью и системным промптом

Теперь создадим класс `Agent`, который умеет:
- хранить историю диалога;
- использовать системный промпт (инструкцию поведения);
- генерировать ответы с учётом всей истории.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

class Agent:
    def __init__(self, name, system_prompt, model_path, max_length=512):
        """
        Инициализация агента.

        Args:
            name (str): имя агента
            system_prompt (str): системный промпт (инструкция поведения)
            model_path (str): путь к модели
            max_length (int): максимальная длина генерируемого ответа
        """
        self.name = name
        self.system_prompt = system_prompt
        self.max_length = max_length

        # ============================================================
        # ЗАДАНИЕ №4
        # Загрузите токенизатор и модель внутри конструктора.
        # Сохраните их в атрибуты self.tokenizer и self.model.
        # Подсказка: используйте те же методы, что и в Задании №1.
        # ============================================================
        
        # >>> ВАШ КОД ЗДЕСЬ (загрузка tokenizer и model) <<<
        
        
        
        # История диалога: список пар (speaker, message)
        self.conversation_history = []

    def say(self, user_message=None):
        """
        Генерирует ответ на основе истории диалога и системного промпта.

        Args:
            user_message (str, optional): сообщение пользователя (если есть)

        Returns:
            str: сгенерированный ответ
        """
        # Формируем текст истории диалога
        history_text = "\n".join([
            f"{speaker}: {message}"
            for speaker, message in self.conversation_history
        ])

        # ============================================================
        # ЗАДАНИЕ №5
        # Если передан user_message, добавьте его в history_text.
        # Формат: "\n{self.name}: {user_message}"
        # ============================================================
        
        # >>> ВАШ КОД ЗДЕСЬ (2-3 строки с if) <<<
        
        
        
        # Собираем полный промпт
        full_prompt = (
            f"Системная инструкция: {self.system_prompt}\n\n"
            f"История диалога:\n{history_text}\n\n"
            f"{self.name} отвечает:"
        )

        # Токенизируем и генерируем
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_length=self.max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=self.tokenizer.eos_token_id
        )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Извлекаем только часть после «отвечает:»
        response_part = response.split("отвечает:")[-1].strip()

        # ============================================================
        # ЗАДАНИЕ №6
        # Добавьте сгенерированный ответ в историю диалога.
        # Формат: (self.name, response_part)
        # ============================================================
        
        # >>> ВАШ КОД ЗДЕСЬ (1 строка) <<<
        
        
        return response_part

## Часть 3. Создание двух агентов с разными ролями

In [ ]:
# ============================================================
# ЗАДАНИЕ №7
# Создайте двух агентов:
# 1. optimist — учёный-оптимист
# 2. skeptic  — учёный-скептик
#
# Используйте класс Agent.
# Системные промпты уже написаны ниже — просто передайте их в конструктор.
# Не забудьте указать model_path.
# ============================================================

model_path = r"ПУТЬ_К_ВАШЕЙ_МОДЕЛИ"   # скопируйте значение из ячейки выше

# Учёный-оптимист
# >>> ВАШ КОД ЗДЕСЬ <<<
optimist = Agent(
    name="Учёный-Оптимист",
    system_prompt=(
        "Ты — учёный-оптимист. Веришь в безграничные возможности ИИ. "
        "Аргументированно доказываешь, что ИИ может и должен заменить учёных. "
        "Говоришь воодушевлённо, используешь примеры достижений ИИ."
    ),
    # model_path=...   <-- раскомментируйте и укажите путь
)

# Учёный-скептик
# >>> ВАШ КОД ЗДЕСЬ <<<
skeptic = Agent(
    name="Учёный-Скептик",
    system_prompt=(
        "Ты — учёный-скептик. Считаешь, что ИИ никогда не заменит человека в науке. "
        "Подчёркиваешь важность человеческого творчества, интуиции и этики. "
        "Приводишь контраргументы к каждому тезису оптимиста."
    ),
    # model_path=...   <-- раскомментируйте и укажите путь
)

## Часть 4. Функция запуска диалога между агентами

In [ ]:
def run_dialogue(agent1, agent2, topic, rounds=5):
    """
    Запускает диалог между двумя агентами.

    Args:
        agent1 (Agent): первый агент
        agent2 (Agent): второй агент
        topic (str): начальная тема обсуждения
        rounds (int): количество раундов (парных обменов)
    """
    print(f"Начинается дискуссия на тему: '{topic}'\n")
    print("-" * 50)

    # Первый агент начинает с заданной темы
    first_response = agent1.say(topic)
    print(f"{agent1.name}: {first_response}")

    for i in range(rounds):
        # ============================================================
        # ЗАДАНИЕ №8
        # Внутри цикла:
        # 1. Получите ответ от agent2 (вызовите agent2.say())
        # 2. Выведите его на экран
        # 3. Получите ответ от agent1
        # 4. Выведите его на экран
        # 5. Напечатайте разделитель "-" * 50
        #
        # Подсказка: посмотрите, как выше вызывается agent1.say(topic)
        # ============================================================
        
        # >>> ВАШ КОД ЗДЕСЬ (4-6 строк) <<<
        
        
        
        
        
        pass  # удалите эту строку, когда напишете код

## Часть 5. Запуск дискуссии

После того как все задания выполнены, запустите ячейку ниже.

In [ ]:
# Задаём тему и запускаем диалог
topic = "Может ли ИИ когда‑нибудь полностью заменить учёных в научных исследованиях?"
run_dialogue(optimist, skeptic, topic, rounds=3)